# MC inclusive-jet JEC check

Compare selected inclusive-jet eta distributions before and after jet energy corrections (JEC) in several reconstructed-jet pT intervals. The raw and corrected inputs are `hRecoInclusiveJetRawPtEtaLabUnflipped` and `hRecoInclusiveJetPtEtaLabUnflipped`. Each histogram is projected in its own pT coordinate, so differences include the jet migration caused by JEC. For each pT interval, the notebook makes a square raw/corrected distribution overlay and a two-panel ratio figure: corrected/raw for both beam directions on top and the Pb-going/p-going double ratio on the bottom. Legends identify every curve and are retained with the canvases for reliable batch rendering.

In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import sys

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / 'CMakeLists.txt').is_file() and (p / 'hist_analysis').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Start Jupyter from the jetAnalysis repository or a subdirectory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root
ROOT = load_root(batch=True)
ROOT.TH1.AddDirectory(False)
ROOT.gStyle.SetOptStat(0)
from hist_analysis.config.files import BASE_DIR
from hist_analysis.python.histogram_io import load_histogram, resolve_direction_file
from hist_analysis.python.histogram_ops import ratio_to_nominal
from hist_analysis.python.projections import project_semantic_th2
from hist_analysis.python.root_style import (
    draw_text_block, save_canvas, set_1d_style, set_pad_style,
    style_single_panel_axes,
)


In [ ]:
# Select the MC production and reconstructed-jet selection to inspect.
GENERATOR = 'embedding'              # 'embedding' or 'pythia'
FILE_STEM = 'jetId'                  # 'jetId', 'trkMax', or 'noSel'
PT_RANGES = ((30., 50.), (50., 80.), (80., 120.), (120., 180.), (180., 300.), (300., 500.))  # half-open intervals
X_AXIS_RANGE = (-4.0, 4.0)           # shared eta range, or None for full axes
OUTPUT_DIR = PROJECT_ROOT / 'hist_analysis' / 'output' / 'mc_jet_JEC_check'
SAVE_PNG = False
DISTRIBUTION_Y_RANGE = None          # e.g. (1e-2, 1e7), or None for automatic
# Independent y-axis ranges for the two ratio panels.
CORRECTED_RAW_Y_RANGE = (0.8, 3.2)
CORRECTED_RAW_DIRECTION_RATIO_Y_RANGE = (0.7, 1.3)
# ROOT TH1::Divide error options: '' = independent errors, 'B' = binomial.
CORRECTED_RAW_RATIO_OPTION = 'B'
CORRECTED_RAW_DIRECTION_RATIO_OPTION = ''
if FILE_STEM not in ('jetId', 'trkMax', 'noSel'):
    raise ValueError("FILE_STEM must be 'jetId', 'trkMax', or 'noSel'")
if X_AXIS_RANGE is not None and X_AXIS_RANGE[0] >= X_AXIS_RANGE[1]:
    raise ValueError('X_AXIS_RANGE must satisfy low < high')
for option_name, option in (
    ('CORRECTED_RAW_RATIO_OPTION', CORRECTED_RAW_RATIO_OPTION),
    ('CORRECTED_RAW_DIRECTION_RATIO_OPTION', CORRECTED_RAW_DIRECTION_RATIO_OPTION),
):
    if option not in ('', 'B'):
        raise ValueError(f"{option_name} must be '' or 'B'")

# These are the two 2D (pT, eta) inputs compared throughout the notebook.
hist_names = {
    'raw': 'hRecoInclusiveJetRawPtEtaLabUnflipped',
    'corrected': 'hRecoInclusiveJetPtEtaLabUnflipped',
}
files = {direction: resolve_direction_file(BASE_DIR, GENERATOR, direction, FILE_STEM)
         for direction in ('Pbgoing', 'pgoing')}
missing = [path for path in files.values() if not path.exists()]
if missing:
    raise FileNotFoundError('Missing MC input file(s):\n' + '\n'.join(map(str, missing)))
histograms = {direction: {kind: load_histogram(path, name) for kind, name in hist_names.items()}
             for direction, path in files.items()}
print('Inputs:')
for direction, path in files.items():
    print(f'  {direction}: {path}')


In [ ]:
# Keep PyROOT objects alive after the input files and local drawing scopes close.
_objects = []

def project(direction, kind, pt_range):
    """Project an eta distribution in the requested reconstructed-pT interval."""
    return project_semantic_th2(
        histograms[direction][kind], 'eta', pt_range,
        name=f'{direction}_{kind}_eta_{pt_range[0]:g}_{pt_range[1]:g}',
    )

def draw_overlay(canvas, hists, annotations, y_title, *, logy=False, y_range=None):
    """Draw styled curves and a persistent legend on one ROOT canvas."""
    canvas.cd()
    # Match the shared notebook geometry so titles and tick labels are not clipped.
    set_pad_style(canvas, grid_x=False, grid_y=False)
    canvas.SetRightMargin(0.05)
    canvas.SetLogy(logy)
    first = True
    automatic_maximum = max(hist.GetMaximum() for hist in hists.values())
    for index, (label, hist) in enumerate(hists.items()):
        set_1d_style(hist, index)
        hist.SetTitle('')
        hist.GetXaxis().SetTitle('#eta_{lab}')
        hist.GetYaxis().SetTitle(y_title)
        style_single_panel_axes(hist)
        if X_AXIS_RANGE is not None:
            hist.GetXaxis().SetRangeUser(*X_AXIS_RANGE)
        if y_range is not None:
            hist.SetMinimum(y_range[0]); hist.SetMaximum(y_range[1])
        elif first and automatic_maximum > 0:
            # Leave an empty upper band for the in-pad legend.
            hist.SetMaximum(automatic_maximum * (20.0 if logy else 1.8))
        hist.Draw('E' if first else 'E SAME')
        first = False
    # The automatic headroom above keeps this in-pad legend clear of points.
    legend_height = 0.055 * len(hists) + 0.025
    legend = ROOT.TLegend(0.58, max(0.52, 0.88 - legend_height), 0.90, 0.88)
    legend.SetBorderSize(0); legend.SetFillStyle(0)
    for label, hist in hists.items():
        legend.AddEntry(hist, label, 'P')
    legend.Draw()
    text_labels = draw_text_block(canvas, annotations, x=0.20, y=0.87,
                                  text_size=0.035, line_spacing=0.045)
    _objects.extend((legend, *text_labels))
    canvas.Update()
    return legend

def draw_comparisons(pt_range):
    """Create the distribution overlay and two-panel ratio figure."""
    tag = f'pt_{pt_range[0]:g}_{pt_range[1]:g}'
    output_prefix = f'{GENERATOR}_{FILE_STEM}'
    raw = {d: project(d, 'raw', pt_range) for d in files}
    corrected = {d: project(d, 'corrected', pt_range) for d in files}
    corrected_raw = {
        direction: ratio_to_nominal(
            corrected[direction], raw[direction],
            name=f'{direction}_corrected_over_raw_{tag}',
            option=CORRECTED_RAW_RATIO_OPTION,
        )
        for direction in files
    }
    direction_ratio = ratio_to_nominal(
        corrected_raw['Pbgoing'], corrected_raw['pgoing'],
        name=f'Pbgoing_over_pgoing_{tag}',
        option=CORRECTED_RAW_DIRECTION_RATIO_OPTION,
    )
    _objects.extend((*raw.values(), *corrected.values(), *corrected_raw.values(), direction_ratio))

    canvas1 = ROOT.TCanvas(f'c_jec_yields_{tag}', '', 800, 800)
    draw_overlay(canvas1, {'Pb-going raw': raw['Pbgoing'], 'Pb-going corrected': corrected['Pbgoing'],
                           'p-going raw': raw['pgoing'], 'p-going corrected': corrected['pgoing']},
                 (f'{GENERATOR.capitalize()} MC',
                  f'{pt_range[0]:g} < p_{{T}}^{{jet}} < {pt_range[1]:g} GeV',
                  'Inclusive jets'),
                 'dN/d#eta_{lab}', logy=True, y_range=DISTRIBUTION_Y_RANGE)
    overlay_name = f'{output_prefix}_overlay_{tag}.pdf'
    save_canvas(canvas1, OUTPUT_DIR / overlay_name, save_png=SAVE_PNG)

    # Ratio diagnostics use a two-panel canvas: JEC response above and
    # the beam-direction comparison below. The legend is placed in the
    # upper part of each pad, away from ratios clustered near unity.
    canvas2 = ROOT.TCanvas(f'c_jec_ratios_{tag}', '', 800, 800)
    top = ROOT.TPad(f'p_top_{tag}', '', 0.0, 0.30, 1.0, 1.0)
    bottom = ROOT.TPad(f'p_bottom_{tag}', '', 0.0, 0.0, 1.0, 0.30)
    top.SetLeftMargin(0.22); top.SetRightMargin(0.05); top.SetTopMargin(0.08); top.SetBottomMargin(0.02)
    bottom.SetLeftMargin(0.22); bottom.SetRightMargin(0.05); bottom.SetTopMargin(0.02); bottom.SetBottomMargin(0.35)
    top.SetGridy(True); bottom.SetGridy(True)
    top.Draw(); bottom.Draw()

    def draw_ratio_panel(pad, hists, y_title, y_range, show_x_title):
        pad.cd()
        first = True
        for index, (label, hist) in enumerate(hists.items()):
            set_1d_style(hist, index)
            hist.SetTitle('')
            hist.GetXaxis().SetTitle('#eta_{lab}' if show_x_title else '')
            hist.GetYaxis().SetTitle(y_title)
            style_single_panel_axes(hist)
            # Compensate for pad height so both y-axis titles have the
            # same visual size in the finished canvas.
            if pad is top:
                hist.GetYaxis().SetTitleSize(0.050)
                hist.GetYaxis().SetLabelSize(0.045)
                hist.GetYaxis().SetTitleOffset(1.55)
                hist.GetXaxis().SetLabelSize(0.0)
            else:
                hist.GetYaxis().SetTitleSize(0.117)
                hist.GetYaxis().SetLabelSize(0.105)
                hist.GetYaxis().SetTitleOffset(0.55)
                hist.GetXaxis().SetTitleSize(0.117)
                hist.GetXaxis().SetLabelSize(0.105)
            if X_AXIS_RANGE is not None:
                hist.GetXaxis().SetRangeUser(*X_AXIS_RANGE)
            hist.SetMinimum(y_range[0]); hist.SetMaximum(y_range[1])
            hist.Draw('P' if first else 'P SAME')
            first = False

    draw_ratio_panel(top, {'Pb-going corrected/raw': corrected_raw['Pbgoing'], 'p-going corrected/raw': corrected_raw['pgoing']},
                     'Corrected / Raw', CORRECTED_RAW_Y_RANGE, False)
    top.cd()
    top_legend = ROOT.TLegend(0.5, 0.45, 0.7, 0.65)
    top_legend.SetBorderSize(0); top_legend.SetFillStyle(0)
    for direction, hist in corrected_raw.items():
        direction_label = 'Pb-going' if direction == 'Pbgoing' else 'p-going'
        top_legend.AddEntry(hist, direction_label, 'P')
    top_legend.Draw()
    annotations = draw_text_block(top, (
        f'{GENERATOR.capitalize()} MC',
        f'{pt_range[0]:g} < p_{{T}}^{{jet}} < {pt_range[1]:g} GeV',
        'JEC comparison',
    ), x=0.24, y=0.82, text_size=0.035, line_spacing=0.045)

    draw_ratio_panel(bottom, {'Pb-going / p-going': direction_ratio},
                     '#frac{C/R (Pb-going)}{C/R (p-going)}',
                     CORRECTED_RAW_DIRECTION_RATIO_Y_RANGE, True)
    bottom.cd()
    # Put the legend on the side with fewer high-valued points.
    eta_axis = direction_ratio.GetXaxis()
    eta_midpoint = 0.5 * (eta_axis.GetXmin() + eta_axis.GetXmax())
    high_threshold = (CORRECTED_RAW_DIRECTION_RATIO_Y_RANGE[0]
                      + 0.55 * (CORRECTED_RAW_DIRECTION_RATIO_Y_RANGE[1]
                                - CORRECTED_RAW_DIRECTION_RATIO_Y_RANGE[0]))
    high_counts = {'left': 0, 'right': 0}
    for bin_index in range(1, direction_ratio.GetNbinsX() + 1):
        if direction_ratio.GetBinContent(bin_index) <= high_threshold:
            continue
        side = 'left' if eta_axis.GetBinCenter(bin_index) < eta_midpoint else 'right'
        high_counts[side] += 1
    if high_counts['left'] <= high_counts['right']:
        bottom_legend_bounds = (0.25, 0.72, 0.55, 0.90)
    else:
        bottom_legend_bounds = (0.60, 0.72, 0.90, 0.90)
    bottom_legend = ROOT.TLegend(*bottom_legend_bounds)
    bottom_legend.SetBorderSize(0); bottom_legend.SetFillStyle(0)
    bottom_legend.AddEntry(direction_ratio, 'Pb-going / p-going', 'P')
    bottom_legend.Draw()
    canvas2.cd(); canvas2.Modified(); canvas2.Update()
    canvas2._ratio_panel_objects = [top, bottom, top_legend, bottom_legend, *annotations]
    ratio_name = f'{output_prefix}_ratio_{tag}.pdf'
    save_canvas(canvas2, OUTPUT_DIR / ratio_name, save_png=SAVE_PNG)
    return canvas1, canvas2

# Generate the distribution and ratio figures for every configured pT interval.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
figures = [draw_comparisons(pt_range) for pt_range in PT_RANGES]
print(f'Wrote {2 * len(figures)} figures to {OUTPUT_DIR}')
